# Pixels to Predictions — Phase 0: Zero-Shot Baseline

**Goal:** Establish baseline accuracy with no training using log-likelihood scoring.

Steps:
1. Mount Drive, setup paths
2. Install packages
3. Load data
4. Load base SmolVLM model
5. Implement log-likelihood scoring
6. Try prompt variants on a small subset
7. Run best prompt on full val set
8. Save results to checkpoint

In [3]:
# ── Cell 1: Mount Drive & Setup ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/kaggle_final_competition")
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"

for d in [CHECKPOINT_DIR, SUBMISSION_DIR]:
    d.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir contents:", sorted(os.listdir(DATA_DIR)))
print("\nSetup complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/kaggle_final_competition
Data dir contents: ['images', 'sample_submission.csv', 'test.csv', 'train.csv', 'val.csv']

Setup complete!


In [4]:
# ── Cell 2: Install Packages ─────────────────────────────────────────────────
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.9 MB/s eta 0:00:00


In [5]:
# ── Cell 3: Imports & Config ─────────────────────────────────────────────────
import json
import random
import time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Config
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE = 224
CHOICE_LETTERS = "ABCDEFGHIJ"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Phase 0 will be very slow on CPU.")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [6]:
# ── Cell 4: Load Data ────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 3109 | Val: 1048 | Test: 1008


In [7]:
# ── Cell 5: Prompt Builders ──────────────────────────────────────────────────
# We define multiple prompt styles to test which works best zero-shot

def build_prompt_default(row, include_answer=False):
    """
    Default style from starter notebook.
    <image>\nContext:\n...\nQuestion: ...\nChoices:\n  A. ...\nAnswer:
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt


def build_prompt_direct(row, include_answer=False):
    """
    Shorter, more direct style — no 'Context' label.
    <image>\n{context}\n\n{question}\nA. ...\nB. ...\nThe answer is
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    if context_str:
        prompt += f"{context_str}\n\n"
    prompt += f"{row['question']}\n{choices_str}\n"
    prompt += "The answer is"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt


def build_prompt_no_context(row, include_answer=False):
    """
    No lecture/hint — just image + question + choices.
    Tests whether context helps or hurts.
    """
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = f"<image>\nQuestion: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt


PROMPT_STYLES = {
    "default": build_prompt_default,
    "direct": build_prompt_direct,
    "no_context": build_prompt_no_context,
}

# Show example of each style
sample_row = train_df.iloc[0]
for name, fn in PROMPT_STYLES.items():
    print(f"\n{'='*60}")
    print(f"Style: {name}")
    print(f"{'='*60}")
    print(fn(sample_row, include_answer=True))


Style: default
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't a

In [8]:
# ── Cell 6: Load Model ───────────────────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("Loading model...")
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pre-compute token IDs for choice letters
choice_token_ids = {}
for letter in CHOICE_LETTERS[:5]:  # A through E
    tokens = processor.tokenizer.encode(letter, add_special_tokens=False)
    choice_token_ids[letter] = tokens[0]
    print(f"  Token for '{letter}': id={tokens[0]}")

print(f"\nModel loaded on: {model.device}")
print(f"Dtype: {dtype}")
print(f"Total params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Loading processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

  Token for 'A': id=49
  Token for 'B': id=50
  Token for 'C': id=51
  Token for 'D': id=52
  Token for 'E': id=53

Model loaded on: cuda:0
Dtype: torch.float16
Total params: 507.5M


In [9]:
# ── Cell 7: Log-Likelihood Scoring Function ─────────────────────────────────

def score_batch_log_likelihood(batch_rows, prompt_fn, batch_size=16):
    """
    Score a batch of questions using log-likelihood of each choice letter.

    For each question:
      1. Build prompt ending with 'Answer:' (or similar)
      2. Forward pass through model
      3. Get logits at the last token position
      4. Extract log-probs for each choice letter (A, B, C, ...)
      5. Predict = argmax of those log-probs

    Returns list of predicted answer indices (0-indexed).
    """
    predictions = []

    for start in range(0, len(batch_rows), batch_size):
        end = min(start + batch_size, len(batch_rows))
        chunk = batch_rows[start:end]

        # Build prompts and load images
        prompts = []
        images = []
        num_choices_list = []

        for _, row in chunk.iterrows():
            prompts.append(prompt_fn(row, include_answer=False))
            img = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            images.append(img)
            num_choices_list.append(int(row["num_choices"]))

        # Tokenize
        inputs = processor(
            text=prompts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

        # Forward pass
        with torch.inference_mode():
            outputs = model(**inputs)

        # Get logits at the last non-padding position for each sample
        logits = outputs.logits  # [batch, seq_len, vocab_size]

        # Find last non-pad token position for each sample
        if "attention_mask" in inputs:
            # Sum attention mask to get sequence lengths
            seq_lengths = inputs["attention_mask"].sum(dim=1) - 1  # 0-indexed
        else:
            seq_lengths = torch.tensor([logits.shape[1] - 1] * logits.shape[0])

        for i in range(len(chunk)):
            last_pos = seq_lengths[i].item()
            last_logits = logits[i, int(last_pos), :]  # [vocab_size]
            log_probs = F.log_softmax(last_logits, dim=-1)

            # Get log-prob for each valid choice letter
            n_choices = num_choices_list[i]
            choice_scores = []
            for j in range(n_choices):
                letter = CHOICE_LETTERS[j]
                token_id = choice_token_ids[letter]
                choice_scores.append(log_probs[token_id].item())

            pred = int(np.argmax(choice_scores))
            predictions.append(pred)

        # Free GPU memory
        del inputs, outputs, logits
        torch.cuda.empty_cache()

    return predictions

print("Log-likelihood scoring function defined.")

Log-likelihood scoring function defined.


In [10]:
# ── Cell 8: Quick Sanity Check (5 samples) ──────────────────────────────────
# Run on 5 val samples to make sure everything works before full eval

sanity_df = val_df.head(5).copy()
preds = score_batch_log_likelihood(sanity_df, build_prompt_default, batch_size=5)

print("Sanity check results:")
print(f"{'ID':<15} {'Predicted':>10} {'Actual':>8} {'Correct':>8}")
print("-" * 45)
for i, (_, row) in enumerate(sanity_df.iterrows()):
    pred = preds[i]
    actual = int(row["answer"])
    correct = "✓" if pred == actual else "✗"
    print(f"{row['id']:<15} {CHOICE_LETTERS[pred]:>10} {CHOICE_LETTERS[actual]:>8} {correct:>8}")

acc = sum(p == int(r["answer"]) for p, (_, r) in zip(preds, sanity_df.iterrows())) / len(preds)
print(f"\nSanity accuracy: {acc:.2%} ({sum(p == int(r['answer']) for p, (_, r) in zip(preds, sanity_df.iterrows()))}/{len(preds)})")

Sanity check results:
ID               Predicted   Actual  Correct
---------------------------------------------
val_00671                A        A        ✓
val_04111                A        B        ✗
val_02022                A        D        ✗
val_01237                D        A        ✗
val_03458                E        E        ✓

Sanity accuracy: 40.00% (2/5)


In [11]:
# ── Cell 9: Prompt Comparison on Small Subset (100 samples) ──────────────────
# Test all 3 prompt styles on 100 val samples to find the best one

subset_df = val_df.head(100).copy()
results = {}

for style_name, prompt_fn in PROMPT_STYLES.items():
    print(f"\nEvaluating prompt style: '{style_name}'...")
    start_time = time.time()

    preds = score_batch_log_likelihood(subset_df, prompt_fn, batch_size=16)

    elapsed = time.time() - start_time
    actuals = [int(row["answer"]) for _, row in subset_df.iterrows()]
    acc = sum(p == a for p, a in zip(preds, actuals)) / len(preds)

    results[style_name] = {
        "accuracy": acc,
        "time_seconds": elapsed,
        "num_samples": len(preds),
    }
    print(f"  Accuracy: {acc:.4f} ({sum(p == a for p, a in zip(preds, actuals))}/{len(preds)})")
    print(f"  Time: {elapsed:.1f}s ({elapsed/len(preds):.2f}s per sample)")

print("\n" + "="*60)
print("PROMPT COMPARISON RESULTS (100 val samples)")
print("="*60)
for name, r in sorted(results.items(), key=lambda x: x[1]["accuracy"], reverse=True):
    print(f"  {name:<15} → accuracy: {r['accuracy']:.4f}  time: {r['time_seconds']:.1f}s")

best_style = max(results, key=lambda x: results[x]["accuracy"])
print(f"\nBest prompt style: '{best_style}' with accuracy {results[best_style]['accuracy']:.4f}")


Evaluating prompt style: 'default'...
  Accuracy: 0.3800 (38/100)
  Time: 143.9s (1.44s per sample)

Evaluating prompt style: 'direct'...
  Accuracy: 0.3200 (32/100)
  Time: 98.9s (0.99s per sample)

Evaluating prompt style: 'no_context'...
  Accuracy: 0.2400 (24/100)
  Time: 94.5s (0.95s per sample)

PROMPT COMPARISON RESULTS (100 val samples)
  default         → accuracy: 0.3800  time: 143.9s
  direct          → accuracy: 0.3200  time: 98.9s
  no_context      → accuracy: 0.2400  time: 94.5s

Best prompt style: 'default' with accuracy 0.3800


In [12]:
# ── Cell 10: Full Val Evaluation with Best Prompt ────────────────────────────
# Run the best prompt style on the entire val set

best_prompt_fn = PROMPT_STYLES[best_style]
print(f"Running full val evaluation with prompt style: '{best_style}'")
print(f"Val set size: {len(val_df)}")

start_time = time.time()
all_preds = score_batch_log_likelihood(val_df, best_prompt_fn, batch_size=16)
elapsed = time.time() - start_time

actuals = [int(row["answer"]) for _, row in val_df.iterrows()]
val_acc = sum(p == a for p, a in zip(all_preds, actuals)) / len(all_preds)

print(f"\n{'='*60}")
print(f"PHASE 0 RESULTS — Zero-Shot Baseline")
print(f"{'='*60}")
print(f"Val Accuracy: {val_acc:.4f} ({sum(p == a for p, a in zip(all_preds, actuals))}/{len(all_preds)})")
print(f"Total time: {elapsed:.1f}s ({elapsed/len(all_preds):.2f}s per sample)")
print(f"Prompt style: {best_style}")
print(f"TA Baseline: 0.6781 (for reference)")

Running full val evaluation with prompt style: 'default'
Val set size: 1048

PHASE 0 RESULTS — Zero-Shot Baseline
Val Accuracy: 0.5563 (583/1048)
Total time: 1394.3s (1.33s per sample)
Prompt style: default
TA Baseline: 0.6781 (for reference)


In [13]:
# ── Cell 11: Per-Category Breakdown ──────────────────────────────────────────
# See which categories the model struggles with

val_df_copy = val_df.copy()
val_df_copy["pred"] = all_preds
val_df_copy["correct"] = val_df_copy["pred"] == val_df_copy["answer"].astype(int)

print("Accuracy by SUBJECT:")
print(val_df_copy.groupby("subject")["correct"].mean().sort_values(ascending=False).to_string())

print("\nAccuracy by GRADE:")
print(val_df_copy.groupby("grade")["correct"].mean().sort_values(ascending=False).to_string())

print("\nAccuracy by NUM_CHOICES:")
print(val_df_copy.groupby("num_choices")["correct"].mean().sort_values(ascending=False).to_string())

print("\nBottom 10 CATEGORIES (hardest):")
cat_acc = val_df_copy.groupby("category").agg(
    accuracy=("correct", "mean"),
    count=("correct", "count")
).sort_values("accuracy")
print(cat_acc.head(10).to_string())

print("\nTop 10 CATEGORIES (easiest):")
print(cat_acc.tail(10).to_string())

Accuracy by SUBJECT:
subject
social science      0.584362
natural science     0.550837
language science    0.464286

Accuracy by GRADE:
grade
grade1     1.000000
grade5     0.680672
grade2     0.611111
grade4     0.598837
grade3     0.556604
grade7     0.554945
grade6     0.526042
grade8     0.489177
grade12    0.250000
grade10    0.000000

Accuracy by NUM_CHOICES:
num_choices
2    0.627049
4    0.563492
3    0.551181
5    0.181818

Bottom 10 CATEGORIES (hardest):
                                     accuracy  count
category                                            
Ancient Egypt and Kush               0.000000      1
Age of Exploration                   0.000000      1
Early Americas                       0.000000      2
Early 19th century American history  0.000000      1
Rome and the Byzantine Empire        0.000000      2
The American Revolution              0.000000      1
Natural resources and human impacts  0.000000      1
Medieval Asia                        0.000000      2
G

In [14]:
# ── Cell 12: Save Phase 0 Results ────────────────────────────────────────────
import json

phase0_results = {
    "phase": "phase0_zero_shot",
    "model": MODEL_ID,
    "best_prompt_style": best_style,
    "prompt_comparison": results,
    "full_val_accuracy": val_acc,
    "full_val_total": len(all_preds),
    "full_val_correct": sum(p == a for p, a in zip(all_preds, actuals)),
    "inference_time_seconds": elapsed,
    "batch_size": 16,
    "per_subject": val_df_copy.groupby("subject")["correct"].mean().to_dict(),
    "per_grade": val_df_copy.groupby("grade")["correct"].mean().to_dict(),
    "per_num_choices": val_df_copy.groupby("num_choices")["correct"].mean().to_dict(),
}

# Save to checkpoint dir
results_path = CHECKPOINT_DIR / "phase0_zero_shot_results.json"
with open(results_path, "w") as f:
    json.dump(phase0_results, f, indent=2)

# Also save per-sample predictions for error analysis later
preds_df = val_df[["id", "answer"]].copy()
preds_df["pred"] = all_preds
preds_df["correct"] = preds_df["pred"] == preds_df["answer"].astype(int)
preds_path = CHECKPOINT_DIR / "phase0_val_predictions.csv"
preds_df.to_csv(preds_path, index=False)

print(f"Results saved to: {results_path}")
print(f"Predictions saved to: {preds_path}")
print(f"\nPhase 0 complete! Zero-shot val accuracy: {val_acc:.4f}")
print(f"Next: Phase 1 — QLoRA fine-tuning to beat this baseline.")

Results saved to: /content/drive/MyDrive/kaggle_final_competition/checkpoints/phase0_zero_shot_results.json
Predictions saved to: /content/drive/MyDrive/kaggle_final_competition/checkpoints/phase0_val_predictions.csv

Phase 0 complete! Zero-shot val accuracy: 0.5563
Next: Phase 1 — QLoRA fine-tuning to beat this baseline.


In [15]:
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"VRAM reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

VRAM used: 1.0 GB
VRAM reserved: 2.1 GB
